# Module 07: Vectors Similarity


# 7.1 Introduction to Word Vectors


## 🔢 What are Word Vectors?

Computers don't understand words like "dog" or "cat". They only understand numbers.

A **word vector** (or embedding) is a multi-dimensional array of numbers that represents a word's semantic meaning. 

If two words have similar meanings (like "dog" and "puppy"), their vectors will be close together in this multi-dimensional mathematical space. If they are unrelated (like "dog" and "calculator"), their vectors will be far apart.

### How are they created?
Word vectors are generated by algorithms like **Word2Vec** or **GloVe**. These algorithms read massive amounts of text (like the entire internet) and look at which words frequently appear near other words. 

*"You shall know a word by the company it keeps." - J.R. Firth*


## ⚠️ CRITICAL SETUP FOR THIS MODULE

Up until now, we have been using the small English model (`en_core_web_sm`). **Small models DO NOT contain word vectors.** 

To use vector similarity in spaCy, you **must** use a medium (`md`) or large (`lg`) model! Let's download the medium model now.


In [1]:
import spacy
from spacy.cli import download

# We MUST use the medium model to get word vectors!
print("Downloading en_core_web_md (This might take a minute, it's ~40MB)...")
download("en_core_web_md")
print("Download complete!")


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Download complete!



<br><br>

---

<br><br>


# 7.2 Accessing Vectors in spaCy


## 📊 Accessing the Vector Array

Once you have loaded a medium or large model, accessing a word's vector is incredibly easy. Just use the `token.vector` property!


In [1]:
import spacy

# Load the medium model!
nlp = spacy.load("en_core_web_md")

doc = nlp("apple")
token = doc[0]

# Print the vector
print(f"Vector for '{token.text}':")
print(token.vector)
print(f"\nVector shape/dimensions: {token.vector.shape}")


Vector for 'apple':
[-0.6334     0.18981   -0.53544   -0.52658   -0.30001    0.30559
 -0.49303    0.14636    0.012273   0.96802    0.0040354  0.25234
 -0.29864   -0.014646  -0.24905   -0.67125   -0.053366   0.59426
 -0.068034   0.10315    0.66759    0.024617  -0.37548    0.52557
  0.054449  -0.36748   -0.28013    0.090898  -0.025687  -0.5947
 -0.24269    0.28603    0.686      0.29737    0.30422    0.69032
  0.042784   0.023701  -0.57165    0.70581   -0.20813   -0.03204
 -0.12494   -0.42933    0.31271    0.30352    0.09421   -0.15493
  0.071356   0.15022   -0.41792    0.066394  -0.034546  -0.45772
  0.57177   -0.82755   -0.27885    0.71801   -0.12425    0.18551
  0.41342   -0.53997    0.55864   -0.015805  -0.1074    -0.29981
 -0.17271    0.27066    0.043996   0.60107   -0.353      0.6831
  0.20703    0.12068    0.24852   -0.15605    0.25812    0.007004
 -0.10741   -0.097053   0.085628   0.096307   0.20857   -0.23338
 -0.077905  -0.030906   1.0494     0.55368   -0.10703    0.052234
  0.4

The medium model uses 300-dimensional vectors, which means the semantic meaning of "apple" is represented by 300 specific numbers!

## 🔍 Vector Properties

You can easily check if a word actually has a vector in the model's vocabulary using `token.has_vector`.


In [3]:
doc2 = nlp("dog cat asdfghjkl")

for token in doc2:
    print(f"{token.text:<10} | Has Vector: {token.has_vector} | Vector Norm: {token.vector_norm:.2f}")


dog        | Has Vector: True | Vector Norm: 7.44
cat        | Has Vector: True | Vector Norm: 7.44
asdfghjkl  | Has Vector: True | Vector Norm: 6.61


Notice that `asdfghjkl` does not have a vector because it's not a real word in the model's training data. This is known as an **Out-Of-Vocabulary (OOV)** word.



<br><br>

---

<br><br>


# 7.3 Computing Semantic Similarity


## 📏 Cosine Similarity

To find out how similar two words are, we measure the angle between their two vectors in the 300-dimensional space. This mathematical measurement is called **Cosine Similarity**.
- `1.0` means identical
- `0.0` means absolutely no relationship
- `-1.0` means exact opposites (though this rarely happens perfectly in practice)

In spaCy, you simply call the `.similarity()` method on any Token, Span, or Doc!


In [1]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_md")

# Let's compare some tokens
token1 = nlp("dog")[0]
token2 = nlp("cat")[0]
token3 = nlp("car")[0]

print(f"Similarity between '{token1.text}' and '{token2.text}': {token1.similarity(token2):.4f}")
print(f"Similarity between '{token1.text}' and '{token3.text}': {token1.similarity(token3):.4f}")


Similarity between 'dog' and 'cat': 1.0000
Similarity between 'dog' and 'car': 0.1930


## 📝 Document and Span Similarity

How do you compare an entire sentence to another sentence? 

By default, spaCy calculates the **average vector** of all the words in the sentence (Doc or Span) and compares those averages.


In [2]:
doc1 = nlp("I like to eat apples.")
doc2 = nlp("I enjoy eating oranges.")
doc3 = nlp("The stock market crashed today.")

print(f"Doc 1 & Doc 2 Similarity: {doc1.similarity(doc2):.4f}")
print(f"Doc 1 & Doc 3 Similarity: {doc1.similarity(doc3):.4f}")


Doc 1 & Doc 2 Similarity: 0.8637
Doc 1 & Doc 3 Similarity: 0.6186


Notice how Doc 1 and Doc 2 have a high similarity score because "like"/"enjoy" and "apples"/"oranges" have similar semantic meanings, even though the exact words are different!



<br><br>

---

<br><br>


# 7.4 Vector Operations & Analogies


## 🧮 Vector Arithmetic

Because vectors are just arrays of numbers, you can add and subtract them. 
The most famous example in NLP is:
`KING - MAN + WOMAN = QUEEN`

Let's see if we can do this using spaCy and NumPy!


In [1]:
import spacy
import numpy as np

nlp = spacy.load("en_core_web_md")

# Get the 300D vectors
king = nlp.vocab["king"].vector
man = nlp.vocab["man"].vector
woman = nlp.vocab["woman"].vector

# Perform the math
result_vector = king - man + woman


## 🎯 Finding the closest word

Now we have a new mathematical vector representing `result_vector`. How do we find out what word this vector represents? 

We have to iterate through the entire vocabulary, compare the `result_vector` to every word using Cosine Similarity, and find the word with the highest score!


In [2]:
from numpy import dot
from numpy.linalg import norm

# Define our own cosine similarity function to use with numpy vectors
def cosine_similarity(v1, v2):
    if norm(v1) == 0 or norm(v2) == 0: return 0.0
    return dot(v1, v2) / (norm(v1) * norm(v2))

# Look for the closest word in a subset of common words to keep it fast
queries = ["queen", "princess", "king", "royal", "girl"]
best_word = None
best_score = -1

for word in queries:
    score = cosine_similarity(result_vector, nlp.vocab[word].vector)
    print(f"Similarity to '{word}': {score:.4f}")
    
    # We ignore the original words we used in the math
    if word not in ["king", "man", "woman"] and score > best_score:
        best_score = score
        best_word = word

print(f"\n=> The closest word to (KING - MAN + WOMAN) is: {best_word.upper()}!")


Similarity to 'queen': 0.4812
Similarity to 'princess': 0.1988
Similarity to 'king': 0.5201
Similarity to 'royal': 0.2023
Similarity to 'girl': 0.7345

=> The closest word to (KING - MAN + WOMAN) is: GIRL!



<br><br>

---

<br><br>


# 7.5 Using Custom Word Vectors


## 📥 Loading External Vectors

Sometimes, spaCy's default vectors aren't perfectly tuned for your domain (like highly specialized medical or legal texts).

You can train your own word vectors using libraries like `Gensim` (Word2Vec/FastText), save them as a text file, and load them into a blank spaCy model!

### The `spacy init vectors` command
If you have a `.txt` file containing custom vectors (Word2Vec format), you can compile them into a spaCy pipeline using the command line:

```bash
python -m spacy init vectors en custom_vectors.txt my_custom_spacy_model
```

Then, you simply load your new model in Python:

```python
nlp_custom = spacy.load("./my_custom_spacy_model")
```


## 🧠 Setting Vectors Manually via Code

You can also inject vectors directly into a vocabulary in Python if you are working dynamically.


In [ ]:
import spacy
import numpy as np

# Create a completely blank English model (no vectors)
nlp_blank = spacy.blank("en")
print(f"Does the blank model have vectors? {nlp_blank.vocab.vectors.shape}")

# Let's manually add a vector for the word 'spaceship'
nlp_blank.vocab.vectors.name = "my_custom_vectors"

# The vector must be added to the Vectors table via hash
word_hash = nlp_blank.vocab.strings.add("spaceship")

# Create a fake 300D vector of random numbers
fake_vector = np.random.uniform(-1, 1, (300,))
# We must resize the empty vectors table before we can add to it
nlp_blank.vocab.vectors.resize((1, 300))
nlp_blank.vocab.vectors.add(word_hash, vector=fake_vector)

print(f"Does the model have vectors now? {nlp_blank.vocab.vectors.shape}")
print(f"Does 'spaceship' have a vector? {nlp_blank('spaceship')[0].has_vector}")



<br><br>

---

<br><br>


# 7.6 Limitations of Static Vectors


## 🛑 The Problem with Word2Vec / GloVe

The vectors we've been using are called **Static Vectors**. This means every word has exactly ONE vector representation.

Let's think about the word **"bank"** in these two sentences:
1. "I deposited money in the **bank**."
2. "I sat on the river **bank**."

In static vectors, both instances of the word "bank" have the *exact same 300 numbers*! The model has averaged out the meaning of a financial institution and a river edge into a single vector. This causes major inaccuracy in semantic similarity.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_md")

# Let's prove it
doc = nlp("I deposited money in the bank. I sat on the river bank.")

bank1 = doc[5]
bank2 = doc[12]

print(f"Token 1: {bank1.text} (Context: Money)")
print(f"Token 2: {bank2.text} (Context: River)")

# Are their vectors perfectly identical?
import numpy as np
are_identical = np.array_equal(bank1.vector, bank2.vector)
print(f"\nAre the vectors for both 'bank's exactly the same? {are_identical}")


## 🤖 The Solution: Transformers (Contextual Vectors)

Modern NLP has solved this problem using **Transformer models** (like BERT). Transformers do not use a fixed dictionary of vectors. Instead, they read the *entire sentence*, look at the surrounding context, and generate a **Contextualized Embedding** dynamically.

This means the "bank" in sentence 1 will have a completely different vector than the "bank" in sentence 2!

We will learn how to use Transformer models in spaCy later in the curriculum (Module 17).


## 🎉 Summary of Part 2 (Core NLP Features)

Congratulations! You have completed **Part 2** of the curriculum.

You now know how to:
- Tokenize and normalize messy text.
- Extract linguistic data (POS tags, Lemmatization, Noun Chunks).
- Build NER systems and visualize entities.
- Use Word Vectors to mathematically compare the meaning of words and documents.

In **Part 3: Pattern Matching & Rules**, we will stop using statistics and learn how to build blazing-fast, rule-based systems to extract exact patterns from text!
